# Chatbot Inteligente con Documentos

Notebook base para Google Colab. Implementa lectura documental, embeddings, FAISS, LangChain y preguntas sobre documentos PDF/TXT/DOCX.

## 1. Instalacion de dependencias

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers pypdf docx2txt

## 2. Carga de documentos

In [ ]:
from google.colab import files
uploaded = files.upload()
list(uploaded.keys())

## 3. Lectura PDF, TXT o DOCX

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader

def load_document(path):
    path = Path(path)
    if path.suffix.lower() == '.pdf':
        return PyPDFLoader(str(path)).load()
    if path.suffix.lower() == '.docx':
        return Docx2txtLoader(str(path)).load()
    if path.suffix.lower() == '.txt':
        return TextLoader(str(path), encoding='utf-8').load()
    raise ValueError('Formato no permitido')

documents = []
for name in uploaded.keys():
    documents.extend(load_document(name))

len(documents)

## 4. Fragmentacion, embeddings y FAISS

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=420, chunk_overlap=80)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

len(chunks)

## 5. Modelo de IA y cadena RAG

In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
generator = pipeline(
    'text2text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=180,
    do_sample=False,
    repetition_penalty=1.2,
    no_repeat_ngram_size=4,
    truncation=True,
)
llm = HuggingFacePipeline(pipeline=generator)

prompt = PromptTemplate(
    input_variables=['context', 'question'],
    template='''Eres un asistente profesional de consulta documental. Responde en espanol claro y formal usando exclusivamente el contexto recuperado. No inventes datos, codigo, librerias ni componentes que no aparezcan en el contexto. Si no hay informacion suficiente, responde: No encuentro esa informacion en los documentos cargados.\n\nContexto:\n{context}\n\nPregunta: {question}\n\nRespuesta:'''
)

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True, output_key='answer')
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={'prompt': prompt}
)

## 6. Preguntas al chatbot

In [ ]:
pregunta = '¿Cual es la idea principal del documento?'
resultado = qa_chain.invoke({'question': pregunta})
print(resultado['answer'])
print('\nFuentes:')
for doc in resultado['source_documents']:
    print(doc.metadata)

## 7. Definiciones solicitadas

**LangChain:** framework que organiza el flujo entre consulta, memoria, recuperacion semantica, documentos y modelo de IA.

**Embeddings:** representaciones numericas del significado del texto; permiten comparar ideas en forma de vectores.

**Busqueda semantica:** recuperacion de informacion por significado, no solo por coincidencia exacta de palabras.